# Transformers:  Attention from Scratch to a Tiny GPT

In the previous notebook, an RNN read a sequence one step at a time, squeezing everything it had seen into a single hidden state.  That design has two weaknesses:
* a bottleneck -> everything about the past must fit in one vector, overwritten at every step, so information from far back fades
* slow training -> step "56" can't be computed until step "55" is done, so the steps can't run in parallel on a GPU

Transformers (from the 2017 paper *Attention Is All You Need*) drop the recurrence.  Every position in a sequence can look directly at every other position and decide which ones matter.

This mechanism is called attention, and it's at the core of every large language model today.

Here's the plan:
1. attention by hand:  the whole idea in a few lines of code
2. learning where to look, and why transformers need positions
3. the transformer block
4. a tiny GPT that writes "Shakespeare"

In [1]:
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("PyTorch", torch.__version__, "  device:", device)

PyTorch 2.15.0   device: mps


# Attention by hand

## Attention is a soft dictionary lookup

A Python dictionary maps keys to values.  Looking up a query returns the value whose key matches it exactly.

Attention does a "soft" version of this:
1. compare the query with *every* key, giving a similarity score for each
   * the score is a dot product, which is larger when the two vectors point in similar directions
2. turn the scores into weights that are positive and add up to 1, using softmax
3. return the weighted average of the values

So instead of returning exactly one value, attention returns a blend.  The blend is dominated by the values whose keys best match the query.

Here's a tiny example with 4 keys and values, each a 2-D vector:

In [4]:
keys = torch.tensor([[1.0, 0.0],
                     [0.0, 1.0],
                     [1.0, 1.0],
                     [-1.0, 0.0]])
values = torch.tensor([[10.0, 0.0],
                       [0.0, 10.0],
                       [5.0, 5.0],
                       [-10.0, 0.0]])
query = torch.tensor([2.0, 0.1])      # points in the direction of the first key

scores = keys @ query                 # one dot product per key
weights = torch.softmax(scores, dim=0)
output = weights @ values             # weighted average of the values

print("scores: ", scores)
print("weights:", weights.round(decimals=3))
print("output: ", output)

scores:  tensor([ 2.0000,  0.1000,  2.1000, -2.0000])
weights: tensor([0.4400, 0.0660, 0.4860, 0.0080])
output:  tensor([6.7498, 3.0890])


The query was most similar to keys 0 and 2, so the output is mostly a mix of values 0 and 2.

## Scaled dot-product attention

In practice, we have many queries at once, stacked as the rows of a matrix $Q$, with keys $K$ and values $V$.  Then all the dot products become one matrix multiplication:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V$$

Here, $d$ is the length of the query and key vectors.

Why divide by $\sqrt{d}$?  It keeps the scores from growing with $d$.  Otherwise, softmax would put nearly all the weight on one key, and its gradients would vanish.

That's the whole mechanism:

In [5]:
def attention(Q, K, V, mask=None):
    # Q: (..., n_queries, d)   K: (..., n_keys, d)   V: (..., n_keys, d_v)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])     # (..., n_queries, n_keys)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))          # blocked positions get weight 0
    weights = torch.softmax(scores, dim=-1)                        # each row sums to 1
    return weights @ V, weights

(We'll explain the `mask` later.)

PyTorch has a fast built-in version, `F.scaled_dot_product_attention`.  Let's check that ours gives the same results, just like we previously checked hand-written gradients against autograd:

In [6]:
torch.manual_seed(0)
Q, K, V = torch.randn(2, 5, 8), torch.randn(2, 7, 8), torch.randn(2, 7, 8)   # batch of 2: 5 queries, 7 keys

ours, weights = attention(Q, K, V)
builtin = F.scaled_dot_product_attention(Q, K, V)
print("output shape:", ours.shape, "  weights shape:", weights.shape)
print("rows of weights sum to 1:", torch.allclose(weights.sum(dim=-1), torch.ones(2, 5)))
print("matches PyTorch:", torch.allclose(ours, builtin, atol=1e-6))

output shape: torch.Size([2, 5, 8])   weights shape: torch.Size([2, 5, 7])
rows of weights sum to 1: True
matches PyTorch: True


## Self-attention

In a transformer, the queries, keys, and values all come from the *same* sequence.  Three learned linear layers turn each position's vector $x_i$ into:
* a query $q_i = W_Q x_i$ -> "what am I looking for?"
* a key $k_i = W_K x_i$ -> "what do I contain?"
* a value $v_i = W_V x_i$ -> "what do I pass on if someone attends to me?"

Every position attends to every position.  The output has the same shape as the input, and each output vector is a blend of information from the whole sequence.

In [7]:
d = 8
W_Q, W_K, W_V = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)

X = torch.randn(1, 6, d)                     # one sequence of 6 positions
out, weights = attention(W_Q(X), W_K(X), W_V(X))
print("input:", X.shape, "  output:", out.shape, "  attention weights:", weights.shape)

input: torch.Size([1, 6, 8])   output: torch.Size([1, 6, 8])   attention weights: torch.Size([1, 6, 6])


# Learning where to look

Here's our task:  each input is 10 random digits followed by a pointer token saying which position to look up.  The answer is the digit at that position.

```
digits at positions 0-9:   3 8 1 1 7 0 4 9 2 6    pointer: "position 4"    ->  answer: 7
```

Tokens are turned into vectors by an embedding.
* `nn.Embedding(n_tokens, d)` is a learnable table with one row of `d` numbers per token
* we use 20 tokens:  digits 0–9 are tokens 0–9, and "position j" is token 10 + j

In [ ]:
N_DIGITS = 10

def make_lookup_batch(batch_size):
    digits = torch.randint(0, 10, (batch_size, N_DIGITS))
    position = torch.randint(0, N_DIGITS, (batch_size,))
    tokens = torch.cat([digits, (10 + position).unsqueeze(1)], dim=1)   # (batch, 11)
    answer = digits[torch.arange(batch_size), position]
    return tokens, answer, position

tokens, answer, position = make_lookup_batch(3)
print(tokens)
print("positions:", position.tolist(), "  answers:", answer.tolist())

Our model will:
* embed the 11 tokens
* let the *last* token (the pointer) attend to all positions
* classify the result into one of 10 digits

The pointer's query has to find the key of the right position.

But there's a catch:  the embedding of the digit 7 is the same wherever the 7 appears.  Nothing says *where* each token is.

So we also add a positional embedding.  This is a second learnable table with one vector per position, added to each token's vector.  The flag `use_positions` lets us turn it off.

In [ ]:
class LookupModel(nn.Module):
    def __init__(self, d=32, use_positions=True):
        super().__init__()
        self.token_embedding = nn.Embedding(20, d)
        self.position_embedding = nn.Embedding(N_DIGITS + 1, d) if use_positions else None
        self.W_Q, self.W_K, self.W_V = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.output = nn.Linear(d, 10)

    def forward(self, tokens):
        h = self.token_embedding(tokens)                         # (batch, 11, d)
        if self.position_embedding is not None:
            h = h + self.position_embedding(torch.arange(tokens.shape[1], device=tokens.device))
        query = self.W_Q(h[:, -1:])                              # only the pointer asks: (batch, 1, d)
        out, weights = attention(query, self.W_K(h), self.W_V(h))
        return self.output(out[:, 0]), weights[:, 0]             # logits (batch, 10), weights (batch, 11)

def train_lookup(model, n_steps=1000):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    for step in range(n_steps + 1):
        tokens, answer, _ = make_lookup_batch(128)
        logits, _ = model(tokens.to(device))
        loss = F.cross_entropy(logits, answer.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 250 == 0:
            accuracy = (logits.argmax(dim=1).cpu() == answer).float().mean()
            print(f"step {step:4d}   loss {loss.item():.3f}   accuracy {accuracy:.3f}")

torch.manual_seed(42)
lookup_model = LookupModel(use_positions=True).to(device)
train_lookup(lookup_model)

Perfect accuracy!

Where does the pointer attend?  The red box marks the position it asks for:

In [ ]:
torch.manual_seed(7)
tokens, answer, position = make_lookup_batch(8)
lookup_model.eval()
with torch.no_grad():
    logits, weights = lookup_model(tokens.to(device))
weights = weights.cpu()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.imshow(weights, cmap="Blues", vmin=0, vmax=1)
for row in range(8):
    for col in range(N_DIGITS):
        ax.text(col, row, tokens[row, col].item(), ha="center", va="center",
                color="white" if weights[row, col] > 0.5 else "black")
    ax.text(N_DIGITS, row, f"p{position[row].item()}", ha="center", va="center")
    ax.add_patch(plt.Rectangle((position[row] - 0.5, row - 0.5), 1, 1, fill=False, ec="red", lw=2))
ax.set_xticks(range(N_DIGITS + 1), list(range(N_DIGITS)) + ["ptr"])
ax.set_xlabel("position"); ax.set_ylabel("example")
ax.set_title("Attention weights of the pointer token (darker = more attention)")
plt.show()

The model puts nearly all its attention on exactly the right position.  Nobody told it to do that -> that's what minimizing the loss produced.

## Without positions

Now let's try the same model *without* positional embeddings:

In [ ]:
torch.manual_seed(42)
no_position_model = LookupModel(use_positions=False).to(device)
train_lookup(no_position_model)

It's stuck far below 100%.  (It's a bit better than the 10% of guessing, because a digit that appears several times is more likely to be the answer.)

Why?  Attention by itself has no idea of order.  It compares vectors with dot products, and a token's vector is the same wherever it sits.

If we shuffle the input positions, the self-attention outputs are just shuffled the same way:

In [ ]:
permutation = torch.tensor([3, 0, 5, 1, 4, 2])
out, _ = attention(W_Q(X), W_K(X), W_V(X))
X_shuffled = X[:, permutation]
out_shuffled, _ = attention(W_Q(X_shuffled), W_K(X_shuffled), W_V(X_shuffled))
print("shuffling the input just shuffles the output:", torch.allclose(out[:, permutation], out_shuffled, atol=1e-6))

For language, order is essential:  "dog bites man" and "man bites dog" contain the same words.  That's why every transformer adds positional information to its inputs.

# The transformer block

A transformer is a stack of identical blocks.  Each block has four ingredients:
1. multi-head attention
2. the causal mask
3. a small feed-forward network
4. residual connections and layer normalization

### 1. Multi-head attention

Instead of one attention step with vectors of size $d$, we run several smaller ones in parallel, called heads, and concatenate their outputs.

Each head can learn to look for something different.  For example, one head might attend to the previous character, while another attends to the start of the word.

In code:
* one linear layer computes all queries, keys, and values
* then we reshape so the heads become an extra batch-like dimension
* our `attention` function can then handle all heads at once

`(batch, time, d)` → `(batch, time, heads, d/heads)` → `(batch, heads, time, d/heads)`

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.qkv = nn.Linear(d, 3 * d)       # queries, keys, and values in one layer
        self.out = nn.Linear(d, d)           # mixes the heads' outputs

    def forward(self, x, mask=None):
        batch, time, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)                          # each (batch, time, d)
        split_heads = lambda t: t.view(batch, time, self.n_heads, d // self.n_heads).transpose(1, 2)
        out, _ = attention(split_heads(q), split_heads(k), split_heads(v), mask)   # (batch, heads, time, d/heads)
        out = out.transpose(1, 2).reshape(batch, time, d)                # heads side by side again
        return self.out(out)

### 2. The causal mask

A language model learns to predict the next token.  At position $t$, it may only use positions $0, \dots, t$.  If it could see position $t+1$, it would just copy the answer!

A causal mask blocks attention to the future.
* it's a lower-triangular matrix, with `True` = allowed
* our `attention` function sets blocked scores to $-\infty$, which softmax turns into a weight of exactly 0

In [ ]:
mask = torch.tril(torch.ones(6, 6, dtype=torch.bool))
print(mask.int())
_, w = attention(W_Q(X), W_K(X), W_V(X), mask)
print(w[0].detach().round(decimals=2))      # row = position, column = position it attends to

### 3. A small feed-forward network

After attention has gathered information from other positions, each position is processed on its own by a small dense network:
* two linear layers with a GELU activation (a smooth version of ReLU)
* a hidden size of $4d$

### 4. Residual connections and layer normalization

Each part *adds* its output to its input:  `x = x + attention(...)`, then `x = x + feed_forward(...)`.
* these residual connections let information and gradients flow straight through a deep stack, which makes deep transformers trainable
* layer normalization standardizes each position's vector before each part, keeping values in a well-behaved range

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attention = MultiHeadSelfAttention(d, n_heads)
        self.norm2 = nn.LayerNorm(d)
        self.feed_forward = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)    # positions exchange information
        x = x + self.feed_forward(self.norm2(x))       # each position is processed on its own
        return x

x = torch.randn(2, 6, 16)
print("input:", x.shape, " output:", TransformerBlock(d=16, n_heads=4)(x).shape)

The output has the same shape as the input, so blocks can be stacked as deep as we like.

A GPT model is:  token embeddings + positional embeddings -> a stack of blocks -> a final layer that predicts the next token.

# A tiny GPT that writes Shakespeare

We'll train a character-level language model:  given some text, predict the next character.
* the training text is about 1 MB of Shakespeare
* each character is a token with an integer ID
* (real language models use subword tokens, such as `"ing"` or `" the"`)

In [ ]:
text_path = Path("datasets/shakespeare.txt")
if not text_path.is_file():
    text_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", text_path)
text = text_path.read_text()

chars = sorted(set(text))
vocab_size = len(chars)
char_to_id = {c: i for i, c in enumerate(chars)}
encode = lambda s: [char_to_id[c] for c in s]
decode = lambda ids: "".join(chars[i] for i in ids)

print(f"{len(text):,} characters, vocabulary of {vocab_size}")
print(text[:150])
print(encode("To be"), "->", decode(encode("To be")))

## Training examples

Each example is a random stretch of `CONTEXT` characters.  The target is the same stretch shifted one character to the left, i.e., at every position, the next character.

So one example trains `CONTEXT` predictions at once, and the causal mask makes sure each position only sees the characters up to itself.

The last 10% of the text is kept for validation.

In [ ]:
data = torch.tensor(encode(text))
n_train = int(0.9 * len(data))
train_data, valid_data = data[:n_train], data[n_train:]
CONTEXT = 64

def get_batch(split, batch_size=32):
    source = train_data if split == "train" else valid_data
    starts = torch.randint(0, len(source) - CONTEXT - 1, (batch_size,))
    x = torch.stack([source[s : s + CONTEXT] for s in starts])
    y = torch.stack([source[s + 1 : s + CONTEXT + 1] for s in starts])
    return x.to(device), y.to(device)

x, y = get_batch("train", batch_size=1)
print("input: ", repr(decode(x[0, :30].tolist())))
print("target:", repr(decode(y[0, :30].tolist())))

## The model

Token embedding + positional embedding -> 4 transformer blocks ($d = 128$, 4 heads) with the causal mask -> layer norm -> a linear layer giving one score per character, at every position.

Technical notes:
* `nn.ModuleList` is a list of layers whose parameters PyTorch can find
* `register_buffer` stores the mask in the model so `.to(device)` moves it too, but it isn't trained

To generate text:
1. predict the next character
2. sample one from the predicted probabilities
3. append it
4. repeat

The temperature divides the scores before the softmax:  below 1 is more conservative, above 1 is more adventurous.

Before training, the output is random:

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, context, d=128, n_heads=4, n_blocks=4):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(context, d)
        self.blocks = nn.ModuleList([TransformerBlock(d, n_heads) for _ in range(n_blocks)])
        self.norm = nn.LayerNorm(d)
        self.output = nn.Linear(d, vocab_size)
        self.register_buffer("mask", torch.tril(torch.ones(context, context, dtype=torch.bool)))

    def forward(self, ids):                                   # ids: (batch, time)
        time = ids.shape[1]
        h = self.token_embedding(ids) + self.position_embedding(torch.arange(time, device=ids.device))
        for block in self.blocks:
            h = block(h, self.mask[:time, :time])
        return self.output(self.norm(h))                      # (batch, time, vocab_size)


@torch.no_grad()
def generate(model, prompt, n_chars=300, temperature=0.8):
    model.eval()
    ids = torch.tensor([encode(prompt)], device=device)
    for _ in range(n_chars):
        logits = model(ids[:, -CONTEXT:])                     # at most the last CONTEXT characters
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)     # sample one character
        ids = torch.cat([ids, next_id], dim=1)
    model.train()
    return decode(ids[0].tolist())


torch.manual_seed(42)
gpt = TinyGPT(vocab_size, CONTEXT).to(device)
print(f"parameters: {sum(p.numel() for p in gpt.parameters()):,}\n")
print(generate(gpt, "ROMEO:\n", n_chars=100))

## Training

The loss is cross-entropy over the vocabulary at every position.
* `F.cross_entropy` wants shape `(n, n_classes)`, so we flatten batch and time together
* we use `AdamW`, the standard optimizer for transformers
* we train for a fixed number of steps (batches)

On a GPU this takes under a minute; on a laptop, a few minutes.

For reference, guessing uniformly among the 65 characters gives a loss of $\log 65 \approx 4.17$.

In [ ]:
@torch.no_grad()
def valid_loss(model, n_batches=20):
    model.eval()
    losses = [F.cross_entropy(model(x).reshape(-1, vocab_size), y.reshape(-1)).item()
              for x, y in (get_batch("valid") for _ in range(n_batches))]
    model.train()
    return sum(losses) / n_batches

N_STEPS = 3000 if device == "cuda" else 1500
optimizer = torch.optim.AdamW(gpt.parameters(), lr=0.003)
torch.manual_seed(42)

for step in range(N_STEPS + 1):
    x, y = get_batch("train")
    loss = F.cross_entropy(gpt(x).reshape(-1, vocab_size), y.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 250 == 0:
        print(f"step {step:5d}   train loss {loss.item():.3f}   valid loss {valid_loss(gpt):.3f}")

In [ ]:
torch.manual_seed(0)
print(generate(gpt, "ROMEO:\n", n_chars=500))

This model knows nothing about words; it predicts one character at a time.  But from the text alone, it learned:
* to spell many words
* to format a play (names in capitals followed by a colon, line breaks)
* some rhythm of the language

And it did this with under a million parameters and a few minutes of training.  It's the same recipe as ChatGPT-style models, at a vastly smaller scale.

Temperature changes the style:

In [ ]:
for temperature in [0.3, 1.5]:
    torch.manual_seed(0)
    print(f"--- temperature {temperature} ---")
    print(generate(gpt, "JULIET:\n", n_chars=200, temperature=temperature), "\n")

Low temperature repeats safe, common patterns.  High temperature makes more mistakes and invents words.

# Scaling up

Nothing in a transformer is specific to language -> any sequence of vectors can go in.

For example, for the ridership forecast, we could:
* replace the token embedding with `nn.Linear(5, d)` applied to each day
* drop the causal mask
* read the forecast from the last position

With only about a thousand training windows, it performs about as well as the GRU.  Transformers shine with lots of data and long sequences.

Real language models use the same blocks, scaled up enormously.  GPT-2 (OpenAI, 2019):
* has 124 million parameters in its smallest version
* uses about 50,000 subword tokens
* was trained on about 40 GB of web text

The optional cell below loads it with Hugging Face (`pip install transformers`, and about 500 MB to download once).

In [ ]:
# Optional: needs `pip install transformers` and an internet connection
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
    gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(device).eval()

    print(tokenizer.tokenize("Transformers are unbelievably useful."))    # Ġ marks a token starting with a space
    print(f"GPT-2 parameters: {sum(p.numel() for p in gpt2.parameters()):,}\n")

    torch.manual_seed(0)
    inputs = tokenizer("The most important idea in machine learning is", return_tensors="pt").to(device)
    generated = gpt2.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=0.8, top_k=50,
                              pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(generated[0]))
except Exception as e:
    print("Skipping GPT-2:", type(e).__name__, str(e)[:200])

# Summary

* attention is a soft dictionary lookup
  * compare a query with every key, turn the scores into weights with softmax, and average the values:  $\text{softmax}(QK^\top/\sqrt{d})\,V$
* in self-attention, every position makes a query, a key, and a value, and attends to every position
  * attention learns where to look
* attention ignores order, so transformers add positional embeddings
* a transformer block = multi-head self-attention + a small feed-forward network, each with a residual connection and layer normalization
  * models are stacks of identical blocks
* a causal mask stops each position from seeing the future, which turns a transformer into a language model
  * generating text = predict, sample, append, repeat